In [ ]:
#1.使用朴素贝叶斯解决20类新闻分类

from sklearn.datasets import fetch_20newsgroups

newsgroups = fetch_20newsgroups(data_home=r'D:/Program/code/python/Jen8/data',subset='all')

# 输出特征和标签
print(newsgroups.data[0])  # 输出第一条文本数据
print(newsgroups.target[0])  # 输出第一条数据的标签（0-19）
print(len(newsgroups.target))  # 输出标签数组的长度（样本总数）
print(newsgroups.target_names)  # 输出20个新闻类别的名称列表
print(type(newsgroups.target))  # 输出标签数组的数据类型
print(type(newsgroups.data))  # 输出文本数据的数据类型
print(min(newsgroups.target))  # 输出标签中的最小类别索引
print(max(newsgroups.target))  # 输出标签中的最大类别索引

From: Mamatha Devineni Ratnam <mr47+@andrew.cmu.edu>
Subject: Pens fans reactions
Organization: Post Office, Carnegie Mellon, Pittsburgh, PA
Lines: 12
NNTP-Posting-Host: po4.andrew.cmu.edu



I am sure some bashers of Pens fans are pretty confused about the lack
of any kind of posts about the recent Pens massacre of the Devils. Actually,
I am  bit puzzled too and a bit relieved. However, I am going to put an end
to non-PIttsburghers' relief with a bit of praise for the Pens. Man, they
are killing those Devils worse than I thought. Jagr just showed you why
he is much better than his regular season stats. He is also a lot
fo fun to watch in the playoffs. Bowman should let JAgr have a lot of
fun in the next couple of games since the Pens are going to beat the pulp out of Jersey anyway. I was very disappointed not to see the Islanders lose the final
regular season game.          PENS RULE!!!


10
18846
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train, X_test, y_train, y_test = train_test_split(newsgroups.data, newsgroups.target, test_size=0.25, random_state=42)

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)

print(X_train_tfidf.shape[1])
print(vectorizer.get_feature_names_out()[100000:100010])

146060
['o_uv' 'o_wcp_' 'o_yd' 'oa' 'oa0' 'oa13' 'oa2' 'oa3' 'oa44o' 'oa4no']


In [ ]:
# 导入朴素贝叶斯分类器和评估指标
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# 将测试集文本转为TF-IDF特征向量
X_test_tfidf = vectorizer.transform(X_test)

# 初始化多项式朴素贝叶斯模型并设置平滑参数alpha=1
nb = MultinomialNB(alpha=1)
# 使用训练集TF-IDF特征和标签训练模型
nb.fit(X_train_tfidf, y_train)

# 对测试集进行预测
y_pred = nb.predict(X_test_tfidf)

# 输出模型在测试集上的准确率
print(accuracy_score(y_test, y_pred))
# 输出详细的分类报告，包括精确率、召回率和F1分数
print(classification_report(y_test, y_pred, target_names=newsgroups.target_names))

0.8425297113752123
                          precision    recall  f1-score   support

             alt.atheism       0.88      0.72      0.79       198
           comp.graphics       0.86      0.79      0.82       245
 comp.os.ms-windows.misc       0.88      0.83      0.85       242
comp.sys.ibm.pc.hardware       0.66      0.86      0.75       238
   comp.sys.mac.hardware       0.95      0.84      0.89       250
          comp.windows.x       0.96      0.80      0.87       260
            misc.forsale       0.96      0.66      0.78       241
               rec.autos       0.89      0.93      0.91       244
         rec.motorcycles       0.91      0.95      0.93       219
      rec.sport.baseball       0.96      0.94      0.95       261
        rec.sport.hockey       0.90      0.98      0.94       245
               sci.crypt       0.78      0.98      0.87       251
         sci.electronics       0.92      0.80      0.86       249
                 sci.med       0.97      0.88      0.92 

In [ ]:
alt_atheism_index = newsgroups.target_names.index('alt.atheism')  # 获取 'alt.atheism' 类别的索引
predicted_as_alt_atheism = (y_pred == alt_atheism_index)  # 预测为 alt.atheism 的布尔数组
actual_alt_atheism = (y_test == alt_atheism_index)  # 实际为 alt.atheism 的布尔数组

TP = ((y_pred == alt_atheism_index) & (y_test == alt_atheism_index)).sum()  # 计算真正例（True Positive）
FP = ((y_pred == alt_atheism_index) & (y_test != alt_atheism_index)).sum()  # 计算假正例（False Positive）

precision = TP / (TP + FP) if (TP + FP) > 0 else 0  # 计算精确率，避免除零
print("Precision for alt.atheism:", precision)  # 输出 alt.atheism 的精确率


Precision for alt.atheism: 0.8827160493827161


In [14]:
from sklearn.metrics import roc_auc_score

alt_atheism_index = newsgroups.target_names.index('alt.atheism')

y_pred_bin = (nb.predict(X_test_tfidf) == alt_atheism_index).astype(int)
y_true_bin = (y_test == alt_atheism_index).astype(int)
print(y_pred_bin)
print(y_true_bin)

# 计算AUC
auc = roc_auc_score(y_true_bin, y_pred_bin)
print("AUC:", auc)


[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
AUC: 0.8590065475311377
